In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=5237c5d4f4039cfa9de997b857ac4ab4a2ce2cda19159de9488019426a070f04
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# Imports
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# ============================================================
# Quantum random bit generator
# ============================================================
# This function does NOT use Python random.
# It creates the state:
# 1/sqrt(2)(|0> + |1>)
# by applying H to |0>, then measures it.
# The result is randomly 0 or 1.

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)

    # Start from |0>, apply H:
    # |0> -> 1/sqrt(2)(|0> + |1>)
    qc.h(0)

    # Measure the qubit.
    # Because of the H gate, result is randomly 0 or 1.
    qc.measure(0, 0)

    tqc = transpile(qc, backend)
    result = backend.run(tqc, shots=1).result()
    counts = result.get_counts()

    bit = list(counts.keys())[0]
    return int(bit)


def quantum_random_bits(n):
    bits = []

    for _ in range(n):
        bits.append(quantum_random_bit())

    return bits

In [4]:
# ============================================================
# Prepare and measure one qubit
# ============================================================
# basis = 0 means Z basis
# basis = 1 means X basis

def measure_qubit(alice_bit, alice_basis, measurement_basis):
    qc = QuantumCircuit(1, 1)

    # -------------------------
    # Prepare the qubit
    # -------------------------

    # If bit is 1, change |0> to |1>
    if alice_bit == 1:
        qc.x(0)

    # If basis is X basis, apply H
    # bit 0 becomes |+>
    # bit 1 becomes |->
    if alice_basis == 1:
        qc.h(0)

    # -------------------------
    # Measure the qubit
    # -------------------------

    # If measuring in X basis, apply H before measurement
    if measurement_basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    tqc = transpile(qc, backend)
    result = backend.run(tqc, shots=1).result()
    counts = result.get_counts()

    measured_bit = list(counts.keys())[0]
    return int(measured_bit)

In [5]:
# BB84 with attacker Eve
# Set number of qubits and threshold

n_qubits = 50

# If error rate is greater than this threshold,
# Alice and Bob report an attack.
threshold = 0.15

In [6]:
# Alice generates random bits and bases

# Alice randomly generates:
# 1. the bits she wants to send
# 2. the bases she uses to encode those bits
#
# 0 = Z basis
# 1 = X basis

alice_bits = quantum_random_bits(n_qubits)
alice_bases = quantum_random_bits(n_qubits)

print("Alice bits:")
print(alice_bits)

print("Alice bases:")
print(alice_bases)

Alice bits:
[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0]
Alice bases:
[0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1]


In [7]:
# Eve chooses random bases and measures Alice's qubits

# Eve intercepts Alice's qubits.
# Eve does not know Alice's bases, so she randomly chooses bases.
# Then Eve measures each qubit.

eve_bases = quantum_random_bits(n_qubits)
eve_results = []

for i in range(n_qubits):
    eve_result = measure_qubit(
        alice_bit=alice_bits[i],
        alice_basis=alice_bases[i],
        measurement_basis=eve_bases[i]
    )

    eve_results.append(eve_result)

print("Eve bases:")
print(eve_bases)

print("Eve results:")
print(eve_results)

Eve bases:
[1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0]
Eve results:
[1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0]


In [8]:
# Bob measures the qubits resent by Eve
# Eve resends qubits to Bob based on her own measurement results.
# Therefore, Bob is not measuring Alice's original qubits directly.
# Bob is measuring the qubits resent by Eve.
#
# Eve's result = the new bit
# Eve's basis = the new basis

bob_bases = quantum_random_bits(n_qubits)
bob_results = []

for i in range(n_qubits):
    bob_result = measure_qubit(
        alice_bit=eve_results[i],
        alice_basis=eve_bases[i],
        measurement_basis=bob_bases[i]
    )

    bob_results.append(bob_result)

print("Bob bases:")
print(bob_bases)

print("Bob results:")
print(bob_results)

Bob bases:
[1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]
Bob results:
[1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0]


In [9]:
# Alice and Bob compare bases using classical public channel
# Alice and Bob publicly compare their bases.
# They do NOT reveal the actual key bits here.
# They only keep positions where Alice's basis and Bob's basis match.

matching_indices = []

sifted_alice_key = []
sifted_bob_key = []

for i in range(n_qubits):
    if alice_bases[i] == bob_bases[i]:
        matching_indices.append(i)
        sifted_alice_key.append(alice_bits[i])
        sifted_bob_key.append(bob_results[i])

print("Matching basis indices:")
print(matching_indices)

print("Sifted Alice key:")
print(sifted_alice_key)

print("Sifted Bob key:")
print(sifted_bob_key)

Matching basis indices:
[2, 3, 4, 6, 9, 14, 15, 16, 17, 18, 19, 22, 26, 27, 28, 30, 32, 35, 36, 39, 40, 45, 46, 47, 48]
Sifted Alice key:
[1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0]
Sifted Bob key:
[0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1]


In [10]:
# Calculate error rate

# Error checking
# If Eve disturbed the qubits, Alice and Bob's sifted keys may differ.

errors = 0

for a, b in zip(sifted_alice_key, sifted_bob_key):
    if a != b:
        errors += 1

if len(sifted_alice_key) > 0:
    error_rate = errors / len(sifted_alice_key)
else:
    error_rate = 0

print("Number of sifted bits:", len(sifted_alice_key))
print("Number of errors:", errors)
print("Error rate:", error_rate)

Number of sifted bits: 25
Number of errors: 7
Error rate: 0.28


In [11]:
# Detect attacker using threshold
# If the error rate is too high, Alice and Bob report an attack.

if error_rate > threshold:
    print("Attack detected!")
else:
    print("No attack detected.")

Attack detected!


In [12]:
# Summary
print("BB84 with attacker summary")
print("=" * 50)

print("Alice bits:        ", alice_bits)
print("Alice bases:       ", alice_bases)
print("Eve bases:         ", eve_bases)
print("Eve results:       ", eve_results)
print("Bob bases:         ", bob_bases)
print("Bob results:       ", bob_results)
print("Matching indices:  ", matching_indices)
print("Alice sifted key:  ", sifted_alice_key)
print("Bob sifted key:    ", sifted_bob_key)
print("Errors:            ", errors)
print("Error rate:        ", error_rate)

if error_rate > threshold:
    print("Final decision: Attack detected.")
else:
    print("Final decision: No attack detected.")

BB84 with attacker summary
Alice bits:         [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0]
Alice bases:        [0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1]
Eve bases:          [1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0]
Eve results:        [1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0]
Bob bases:          [1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]
Bob results:        [1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0